các bước train mô hình: khai báo thư viện > load base model > Add Layers to Model > Compile Model > Data Transforms > Load Dataset > setup Early Stop > Train > mở khóa trọng số để fine tune > train tiếp cho đến khi đạt Early stop

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!cd /content/drive/MyDrive/DeepLearning

**Khai báo thư viện**

In [ ]:
import torch
import torchvision
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import glob
from PIL import Image
import sys

# dán đường dẫn đến file utils vào đây nếu dùng google colab
#sys.path.append('/content/drive/MyDrive/DeepLearning')
import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

In [ ]:
import torch

# Giải phóng cache đang chiếm VRAM
torch.cuda.empty_cache()

# Thu hồi bộ nhớ không còn dùng
torch.cuda.ipc_collect()


## Load ImageNet Base Model

In [ ]:
# 7.2 Load EfficientNetB2 (pretrained on ImageNet)
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

weights = EfficientNet_B2_Weights.IMAGENET1K_V1
base_model = efficientnet_b2(weights=weights)

print(base_model)


## Freeze Base Model

Đóng băng toàn bộ trọng số của base model

In [ ]:
# 7.3 Freeze base model
base_model.requires_grad_(False)

# Kiểm tra một tham số bất kỳ
next(iter(base_model.parameters())).requires_grad


## Add Layers to Model

In [ ]:
N_CLASSES = 100  # CIFAR-100

my_model = nn.Sequential(
    base_model.features,       # trích xuất đặc trưng
    base_model.avgpool,        # # Gom trung bình toàn cục (Global Avg Pool)
    nn.Flatten(),              # Làm phẳng tensor
    nn.Dropout(p=0.3),         # Chống overfitting
    nn.Linear(1408, 512),      # thêm lớp trung gian
    nn.ReLU(),                 # kích hoạt phi tuyến
    nn.Linear(512, N_CLASSES)  # Lớp đầu ra (softmax 100 chiều)
)

my_model = my_model.to(device)
print(my_model)


## Compile Model

In [ ]:
# 7.5 Compile Model
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(my_model.parameters(), lr=1e-3) #learning rate


my_model = my_model.to(device)

print("Model compiled and ready for training.")


## Data Transforms

In [ ]:
# 7.6 Data Transforms

pre_trans = transforms.Compose([
    transforms.Resize((128, 128)), # Phải dùng 128x128 để khớp với training
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

IMG_WIDTH, IMG_HEIGHT = (128, 128)

# Transform cho tập train (augment dữ liệu)
random_trans = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


## Load Dataset

In [ ]:
import os
from torchvision import datasets

# 7.7 Load CIFAR-100 với transform
BATCH_SIZE = 64
DATA_PATH = "./data"

# Kiểm tra đã có file CIFAR-100 chưa
download_flag = not os.path.exists(os.path.join(DATA_PATH, "cifar-100-python"))

train_data = datasets.CIFAR100(
    root=DATA_PATH, train=True, download=download_flag, transform=random_trans
)
test_data = datasets.CIFAR100(
    root=DATA_PATH, train=False, download=download_flag, transform=pre_trans
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Số ảnh train:", len(train_data))
print("Số ảnh test:", len(test_data))
print("Số batch train:", len(train_loader))
print("Số batch test:", len(test_loader))




## Train the Model


Định nghĩa Lớp EarlyStopper và Setup



In [ ]:
# %%
import torch
import torch.optim as optim
import numpy as np

# >>> BƯỚC KHẮC PHỤC BẮT BUỘC <<<
import importlib
import utils # Giả định bạn đã import utils ở đầu script

# Lệnh này buộc Python tải lại file utils.py, khắc phục lỗi NoneType
importlib.reload(utils) 
# >>> KẾT THÚC BƯỚC KHẮC PHỤC <<<


# --- Định nghĩa Lớp EarlyStopper (Có Checkpoint) ---
class EarlyStopper:
    def __init__(self, patience=7, min_delta=0.001, path='best_checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        # Theo dõi val_loss (càng nhỏ càng tốt)
        if self.best_score is None:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
        elif val_loss > self.best_score - self.min_delta:
            # Loss không cải thiện đáng kể
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            # Loss cải thiện
            print(f'Loss giảm ({self.best_score:.6f} -> {val_loss:.6f}). Lưu Checkpoint...')
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)

# --- Khởi tạo Early Stopper và Tham số ---

BEST_MODEL_PATH = 'best_effnetB2_cifar100.pth'

early_stopper = EarlyStopper(
    patience=3, 
    min_delta=0, 
    path=BEST_MODEL_PATH
)

print(f"Setup EarlyStopper hoàn tất. Tên file lưu checkpoint: {BEST_MODEL_PATH}")

In [ ]:
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import torch.optim as optim

# --- Vòng lặp Huấn luyện ---
for epoch in range(epochs): 
    print('-------------------------------------------')
    print(f'Epoch: {epoch}')

    # Train 
    utils.train(
        my_model, train_loader, len(train_loader.dataset), 
        optimizer, loss_function, device
    )
    
    # Validate 
    val_loss, val_acc = utils.validate(
        my_model, test_loader, len(test_loader.dataset), 
        loss_function, device
    )
    
    # KIỂM TRA EARLY STOPPING VÀ LƯU CHECKPOINT TỐT NHẤT
    early_stopper(val_loss, my_model)
    
    if early_stopper.early_stop:
        print("\n\n################################################################")
        print(f"ĐÃ KÍCH HOẠT EARLY STOPPING tại Epoch {epoch}. Dừng huấn luyện.")
        print("################################################################\n")
        break


Tải và Lưu Trọng số Tốt nhất (Final Save)


In [ ]:
# %%
import torch

# Cấu hình đường dẫn
BEST_CHECKPOINT_PATH = 'best_effnetB2_cifar100.pth'
FINAL_SAVE_PATH = "efficientnetB2_cifar100_FINAL.pth" 

try:
    # 1. Tải lại trọng số TỐT NHẤT từ file checkpoint
    best_model_weights = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    
    # 2. Áp dụng trọng số tốt nhất vào mô hình (ghi đè lên trọng số cuối cùng)
    my_model.load_state_dict(best_model_weights)
    
    # 3. Lưu mô hình tốt nhất với tên cuối cùng
    torch.save(my_model.state_dict(), FINAL_SAVE_PATH) 
    
    # Lấy val_loss tốt nhất
    val_loss_best = early_stopper.best_score
    
    print("--------------------------------------------------")
    print(f"Đã tải lại trọng số TỐT NHẤT (Val Loss: {val_loss_best:.4f})")
    print(f"Mô hình TỐT NHẤT đã được lưu tại: {FINAL_SAVE_PATH}")
    print("--------------------------------------------------")
    
except Exception as e:
    print(f"Lỗi: Không thể tải hoặc lưu mô hình. Nguyên nhân: {e}")

In [ ]:

# Mở khóa toàn bộ model để fine-tune
for param in my_model.parameters():
    param.requires_grad = True
optimizer = Adam(my_model.parameters(), lr=1e-5, weight_decay=1e-4) #learning rate, thêm weight decay để tránh overfit

#sau khi mở khóa thì train lại 

## Evaluate the Model

In [ ]:
# load model đã lưu (hoặc có sẵn)
LOAD_PATH = "efficientnetB2_cifar100_FINAL.pth" # Tên file FINAL đã được lưu
state_dict = torch.load(LOAD_PATH, map_location=device)
my_model.load_state_dict(state_dict)
print(f"Đã tải trọng số thành công từ file: {LOAD_PATH}")

Đã tải trọng số thành công từ file: efficientnetB2_cifar100_FINAL.pth


In [112]:
def evaluate_model(model, dataloader, device):
    model.eval()
    model.to(device)   # đảm bảo model nằm trên GPU/CPU đúng

    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs, labels = imgs.to(device), labels.to(device)  # đưa dữ liệu lên device

            outputs = model(imgs)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    acc = 100. * correct / total
    print(f"Độ chính xác trên tập test: {acc:.2f}%")
    return

evaluate_model(my_model, test_loader, device)

Độ chính xác trên tập test: 83.25%


In [ ]:
save_path = "efficientnetB2_cifar100.pth" 
torch.save(my_model.state_dict(), save_path) 
print(f" Model đã được lưu tại: {save_path}")
#chỉ lưu khi cần thiết

